# Collaboration Network Analysis

## Purpose
Map cross-functional connections and identify collaboration patterns, key connectors, and silos.

## Key Metrics
- **Collaboration Score**: Weighted measure of interaction frequency and quality
- **Network Centrality**: Identify key connectors in the organization
- **Cross-Department Collaboration Rate**: Frequency of inter-team work
- **Collaboration Silos**: Teams with limited external connections

## Research Foundation
- Social network analysis (Borgatti & Everett, 2006)
- Cross-functional team effectiveness (Edmondson, 2012)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load datasets
if not os.path.exists('../data/collaboration_network.csv'):
    print("Generating sample data...")
    exec(open('../data/generate_sample_data.py').read())
else:
    print("Loading existing data...")

org_hierarchy = pd.read_csv('../data/org_hierarchy.csv')
collaboration = pd.read_csv('../data/collaboration_network.csv')
dept_metrics = pd.read_csv('../data/department_metrics.csv')

print(f"\nLoaded {len(org_hierarchy)} employees")
print(f"Loaded {len(collaboration)} collaboration relationships")

## 1. Overall Collaboration Patterns

Assess collaboration frequency and quality across the organization.

In [ ]:
print(f"{'='*80}")
print(f"COLLABORATION OVERVIEW")
print(f"{'='*80}")

# Overall statistics
total_relationships = len(collaboration)
avg_connections = org_hierarchy['unique_connections'].mean()
avg_interactions = collaboration['interaction_count'].mean()
avg_quality = collaboration['avg_interaction_quality'].mean()

print(f"\nCollaboration Metrics:")
print(f"  Total collaboration relationships: {total_relationships}")
print(f"  Average connections per employee:  {avg_connections:.1f}")
print(f"  Average interactions per relationship: {avg_interactions:.1f}")
print(f"  Average interaction quality: {avg_quality:.1f}/5.0")

# Intra vs cross-department
intra_dept = collaboration[collaboration['collaboration_type'] == 'Intra-Department']
cross_dept = collaboration[collaboration['collaboration_type'] == 'Cross-Department']

print(f"\nCollaboration Types:")
print(f"  Intra-department: {len(intra_dept)} ({len(intra_dept)/total_relationships*100:.1f}%)")
print(f"  Cross-department: {len(cross_dept)} ({len(cross_dept)/total_relationships*100:.1f}%)")

# Quality comparison
intra_quality = intra_dept['avg_interaction_quality'].mean()
cross_quality = cross_dept['avg_interaction_quality'].mean()

print(f"\nCollaboration Quality:")
print(f"  Intra-department: {intra_quality:.2f}/5.0")
print(f"  Cross-department: {cross_quality:.2f}/5.0")

if cross_quality < intra_quality - 0.3:
    print(f"  ⚠ Cross-department collaboration quality is significantly lower")
    print(f"     → May indicate silos or communication challenges")

print(f"\n{'='*80}")

# Visualize collaboration patterns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Distribution of connections per employee
ax1.hist(org_hierarchy['unique_connections'], bins=20, 
            alpha=0.7, edgecolor='black', color='steelblue')
ax1.axvline(x=avg_connections, color='red', linestyle='--', linewidth=2, 
               label=f'Avg ({avg_connections:.1f})')
ax1.set_xlabel('Number of Unique Connections', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Employees', fontsize=11, fontweight='bold')
ax1.set_title('Distribution of Network Connections', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Intra vs Cross collaboration comparison
collab_types = ['Intra-\nDepartment', 'Cross-\nDepartment']
counts = [len(intra_dept), len(cross_dept)]
qualities = [intra_quality, cross_quality]

x = np.arange(len(collab_types))
width = 0.35

bars1 = ax2.bar(x - width/2, counts, width, label='Count', alpha=0.8, color='steelblue', edgecolor='black')
ax2_twin = ax2.twinx()
bars2 = ax2_twin.bar(x + width/2, qualities, width, label='Quality', alpha=0.8, color='coral', edgecolor='black')

ax2.set_ylabel('Number of Relationships', fontsize=11, fontweight='bold')
ax2.set_xlabel('Collaboration Type', fontsize=11, fontweight='bold')
ax2_twin.set_ylabel('Avg Quality (1-5)', fontsize=11, fontweight='bold')
ax2.set_title('Collaboration Type Comparison', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(collab_types)
ax2_twin.set_ylim(0, 5)
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Key Connectors Identification

Identify employees who are critical hubs in the collaboration network.

In [ ]:
print(f"{'='*80}")
print(f"KEY CONNECTORS ANALYSIS")
print(f"{'='*80}")

# Top connectors by collaboration score
top_connectors = org_hierarchy.nlargest(10, 'collaboration_score')[[
    'employee_id', 'level', 'department', 'unique_connections', 
    'cross_dept_interactions', 'collaboration_score'
]]

print(f"\nTop 10 Key Connectors:")
print(top_connectors.to_string(index=False))

# Analyze connector characteristics
high_connectors = org_hierarchy[org_hierarchy['collaboration_score'] > 
                                 org_hierarchy['collaboration_score'].quantile(0.75)]

print(f"\n📊 HIGH CONNECTOR PROFILE (Top 25%):")
print(f"  Count: {len(high_connectors)}")
print(f"  Average connections: {high_connectors['unique_connections'].mean():.1f}")
print(f"  Avg cross-dept interactions: {high_connectors['cross_dept_interactions'].mean():.1f}")

# Level distribution
connector_levels = high_connectors['level'].value_counts()
print(f"\n  By Level:")
for level, count in connector_levels.head(5).items():
    pct = count / len(high_connectors) * 100
    print(f"    {level:15s}: {count} ({pct:.1f}%)")

# Department distribution
connector_depts = high_connectors['department'].value_counts()
print(f"\n  By Department:")
for dept, count in connector_depts.head(5).items():
    pct = count / len(high_connectors) * 100
    print(f"    {dept:15s}: {count} ({pct:.1f}%)")

print(f"\n⚠ KEY CONNECTOR RISK:")
print(f"  These {len(high_connectors)} employees are critical to information flow")
print(f"  Loss of key connectors can fragment the organization")
print(f"  → Ensure retention of high connectors")
print(f"  → Develop backup connections to reduce single points of failure")

print(f"\n{'='*80}")

# Visualize connector distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: Connections vs Cross-dept interactions
scatter = ax1.scatter(org_hierarchy['unique_connections'], 
                     org_hierarchy['cross_dept_interactions'],
                     c=org_hierarchy['collaboration_score'], 
                     s=50, alpha=0.6, cmap='viridis', edgecolors='black', linewidth=0.5)

ax1.set_xlabel('Unique Connections', fontsize=11, fontweight='bold')
ax1.set_ylabel('Cross-Department Interactions', fontsize=11, fontweight='bold')
ax1.set_title('Network Connectivity Map', fontsize=13, fontweight='bold')
ax1.grid(alpha=0.3)

cbar = plt.colorbar(scatter, ax=ax1)
cbar.set_label('Collaboration Score', fontsize=10, fontweight='bold')

# Top connectors by department
dept_top_connectors = org_hierarchy.groupby('department')['collaboration_score'].mean().sort_values(ascending=False)

bars = ax2.barh(dept_top_connectors.index, dept_top_connectors.values, 
                alpha=0.7, edgecolor='black', color='coral')
ax2.set_xlabel('Average Collaboration Score', fontsize=11, fontweight='bold')
ax2.set_title('Department Collaboration Scores', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

# Add value labels
for i, (dept, score) in enumerate(dept_top_connectors.items()):
    ax2.text(score + 1, i, f'{score:.1f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Cross-Department Collaboration Matrix

Identify which departments collaborate most/least effectively.

In [ ]:
print(f"{'='*80}")
print(f"CROSS-DEPARTMENT COLLABORATION MATRIX")
print(f"{'='*80}")

# Create collaboration matrix
cross_dept_only = collaboration[collaboration['collaboration_type'] == 'Cross-Department']

# Count collaborations between each pair of departments
collab_matrix = cross_dept_only.groupby(['department_1', 'department_2']).agg({
    'interaction_count': 'sum',
    'avg_interaction_quality': 'mean'
}).reset_index()

# Create pivot table for heatmap
departments = sorted(org_hierarchy['department'].unique())
interaction_matrix = pd.DataFrame(0, index=departments, columns=departments)
quality_matrix = pd.DataFrame(0.0, index=departments, columns=departments)

for _, row in collab_matrix.iterrows():
    dept1, dept2 = row['department_1'], row['department_2']
    interactions = row['interaction_count']
    quality = row['avg_interaction_quality']
    
    interaction_matrix.loc[dept1, dept2] += interactions
    interaction_matrix.loc[dept2, dept1] += interactions
    
    quality_matrix.loc[dept1, dept2] = quality
    quality_matrix.loc[dept2, dept1] = quality

# Set diagonal to NaN
np.fill_diagonal(interaction_matrix.values, np.nan)
np.fill_diagonal(quality_matrix.values, np.nan)

# Identify strongest and weakest pairs
collab_matrix_sorted = collab_matrix.sort_values('interaction_count', ascending=False)

print(f"\nTop 5 Strongest Collaboration Pairs:")
for _, row in collab_matrix_sorted.head(5).iterrows():
    print(f"  • {row['department_1']:15s} ↔ {row['department_2']:15s}: "
          f"{row['interaction_count']:.0f} interactions (quality: {row['avg_interaction_quality']:.1f}/5)")

print(f"\nBottom 5 Weakest Collaboration Pairs:")
for _, row in collab_matrix_sorted.tail(5).iterrows():
    print(f"  • {row['department_1']:15s} ↔ {row['department_2']:15s}: "
          f"{row['interaction_count']:.0f} interactions (quality: {row['avg_interaction_quality']:.1f}/5)")

# Identify silos
dept_cross_collab = org_hierarchy.groupby('department')['cross_dept_interactions'].mean().sort_values()

print(f"\n⚠ POTENTIAL SILOS (Low Cross-Department Collaboration):")
silos = dept_cross_collab[dept_cross_collab < dept_cross_collab.median()]
for dept, score in silos.items():
    print(f"  • {dept:20s}: {score:.1f} avg cross-dept interactions")
    print(f"      → Encourage cross-functional projects and relationships")

print(f"\n{'='*80}")

# Visualize collaboration matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap of interaction counts
sns.heatmap(interaction_matrix, annot=True, fmt='.0f', cmap='YlOrRd', 
            ax=ax1, cbar_kws={'label': 'Total Interactions'})
ax1.set_title('Cross-Department Interaction Volume', fontsize=13, fontweight='bold')
ax1.set_xlabel('')
ax1.set_ylabel('')

# Heatmap of quality scores
sns.heatmap(quality_matrix, annot=True, fmt='.1f', cmap='RdYlGn', 
            vmin=1, vmax=5, ax=ax2, cbar_kws={'label': 'Avg Quality (1-5)'})
ax2.set_title('Cross-Department Collaboration Quality', fontsize=13, fontweight='bold')
ax2.set_xlabel('')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Collaboration Improvement Recommendations

Targeted actions to improve cross-functional collaboration.

In [ ]:
print(f"{'='*80}")
print(f"COLLABORATION IMPROVEMENT RECOMMENDATIONS")
print(f"{'='*80}")

# Recommendation 1: Bridge silos
print(f"\n1. BREAK DOWN SILOS")
print(f"{'='*80}")

for dept in silos.index:
    # Find departments this silo needs to connect with
    dept_collabs = collab_matrix[
        (collab_matrix['department_1'] == dept) | (collab_matrix['department_2'] == dept)
    ].sort_values('interaction_count')
    
    if len(dept_collabs) > 0:
        weakest = dept_collabs.iloc[0]
        other_dept = weakest['department_2'] if weakest['department_1'] == dept else weakest['department_1']
        
        print(f"\n  {dept}:")
        print(f"    → Weakest connection: {other_dept} ({weakest['interaction_count']:.0f} interactions)")
        print(f"    → Action: Create cross-functional projects or working groups")
        print(f"    → Assign liaisons between {dept} and {other_dept}")

# Recommendation 2: Leverage key connectors
print(f"\n\n2. LEVERAGE KEY CONNECTORS")
print(f"{'='*80}")

print(f"\nKey connectors can facilitate collaboration:")
for _, connector in top_connectors.head(5).iterrows():
    print(f"  • {connector['level']:15s} in {connector['department']:15s}")
    print(f"      {connector['unique_connections']:.0f} connections, "
          f"{connector['cross_dept_interactions']:.0f} cross-dept interactions")
    print(f"      → Role: Informal ambassador for cross-team initiatives")

print(f"\n  Actions:")
print(f"    • Formally recognize and reward connector roles")
print(f"    • Protect connector time (avoid overloading with meetings)")
print(f"    • Create mentor programs pairing connectors with isolated employees")

# Recommendation 3: Improve collaboration quality
print(f"\n\n3. IMPROVE COLLABORATION QUALITY")
print(f"{'='*80}")

low_quality_pairs = collab_matrix[collab_matrix['avg_interaction_quality'] < 3.5].sort_values('avg_interaction_quality')

if len(low_quality_pairs) > 0:
    print(f"\nLow-quality collaboration pairs (quality <3.5):")
    for _, pair in low_quality_pairs.head(3).iterrows():
        print(f"  • {pair['department_1']:15s} ↔ {pair['department_2']:15s}: "
              f"Quality = {pair['avg_interaction_quality']:.1f}/5")
        print(f"      → Action: Facilitate alignment sessions and shared goals")

print(f"\n  General actions:")
print(f"    • Establish clear cross-functional workflows and handoffs")
print(f"    • Create shared success metrics for cross-team projects")
print(f"    • Run retrospectives to identify collaboration friction points")

# ROI estimate
print(f"\n\n💰 ESTIMATED IMPACT")
print(f"{'='*80}")

current_cross_collab_rate = len(cross_dept_only) / len(collaboration) * 100
target_cross_collab_rate = 35  # Target 35% cross-dept

print(f"\nCurrent cross-department collaboration: {current_cross_collab_rate:.1f}%")
print(f"Target cross-department collaboration: {target_cross_collab_rate:.1f}%")
print(f"\nExpected benefits from improved collaboration:")
print(f"  • Faster decision-making (reduced silos)")
print(f"  • Increased innovation (diverse perspectives)")
print(f"  • Better employee engagement (broader connections)")
print(f"  • Reduced duplicate work (improved coordination)")

print(f"\n{'='*80}")

## Key Takeaways

1. **Key connectors are critical**: Small group of high-connectors facilitate most information flow
2. **Cross-functional collaboration varies**: Some department pairs work well together, others are siloed
3. **Quality matters**: Frequency of interaction doesn't guarantee effective collaboration
4. **Silos are measurable**: Can identify and target specific teams for intervention

**Recommended Next Steps**:
- Break down identified silos through cross-functional projects
- Recognize and protect key connectors
- Improve collaboration quality for low-scoring department pairs
- Target >35% cross-department collaboration rate
- Create formal mechanisms to facilitate cross-functional work